# Laboratorium 1: prawdopodobieństwo warunkowe i ryzyko w symulacji

Karta laboratorium: <https://rodakt.github.io/MAD/laby/01-ryzyko-symulacja.html>

Treść zadań jest w książce, tutaj jest tylko szkielet kodu. Miejsca do uzupełnienia oznaczono `...`.
Notebook otwarty z linku na karcie laboratorium nie jest zapisany na Twoim Dysku: po ponownym kliknięciu linku zobaczysz wersję wyjściową.
Na początku zajęć zapisz kopię (*Plik → Zapisz kopię na Dysku*) i pracuj na niej.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

## Część 0. Rozgrzewka: losowanie i maski logiczne

Porównanie tablicy z liczbą daje tablicę wartości `True`/`False` (maskę).
Średnia z maski to frakcja wartości `True`, a maska wstawiona w nawiasy wybiera elementy.

In [ ]:
x = rng.integers(1, 7, size=20)   # 20 rzutów kostką
print(x)
print(x >= 5)                     # maska
print(np.mean(x >= 5))            # frakcja rzutów 5 lub 6
print(x[x >= 5])                  # tylko te rzuty

## Część 1. Ćwiczenie 1.8 w symulacji

Na tablicy wyznaczyliśmy $p(x)$, $\eta(x)$ i błędy klasyfikatorów, znając rozkład.
W praktyce rozkładu nie znamy, a mamy tylko próbę. Tu losujemy próbę z rozkładu z ćwiczenia
i sprawdzamy, jak blisko wyników z tablicy są oszacowania z próby.

Wpisz wyniki z tablicy (punkt a).

In [ ]:
p_x = np.array([..., ..., ...])   # p(1), p(2), p(3)
eta = np.array([..., ..., ...])   # eta(1), eta(2), eta(3)

### 1.1. Losowanie próby

Rozkład łączny to $p(x)\,p(y\mid x)$, więc losujemy w dwóch krokach: najpierw $X$ z rozkładu $p(x)$,
potem $Y=1$ z prawdopodobieństwem $\eta(X)$. Tablica `eta` jest indeksowana od 0, stąd `eta[X - 1]`.

In [ ]:
def losuj(n):
    X = rng.choice([1, 2, 3], size=n, p=p_x)          # X z rozkładu brzegowego
    Y = (rng.random(n) < eta[X - 1]).astype(int)       # Y = 1 z prawdopodobieństwem eta(X)
    return X, Y

X, Y = losuj(10)
print(X)
print(Y)

### 1.2. Szacowanie $p(x)$ i $\eta(x)$ z próby

Liczby $p(x)$ i $\eta(x)$ z tablicy opisują rozkład, czyli populację: są stałe, ale w praktyce ich nie znamy.
Znamy tylko próbę, więc zastępujemy je oszacowaniami $\widehat p(x)$ i $\widehat\eta(x)$ wyznaczonymi z danych
(daszek oznacza wielkość obliczoną z próby). Tak powstaje każdy klasyfikator uczony z danych:
w regule $\eta(x)>1/2$ nieznane $\eta$ zastępujemy przez $\widehat\eta$ (zob. punkt 1.4).

Dlaczego frakcje? W interpretacji częstościowej prawdopodobieństwo zdarzenia to częstość, z jaką występuje
ono w długim ciągu niezależnych powtórzeń (formalnie gwarantuje to prawo wielkich liczb).
Stąd $\widehat p(x)$ to frakcja obserwacji z $X=x$. Prawdopodobieństwo warunkowe jest częstością w podpróbie:
$$
\eta(x)=\frac{p(x,1)}{p(x)}\approx\frac{n(x,1)/n}{n(x)/n}=\frac{n(x,1)}{n(x)},
$$
gdzie $n(x)$ to liczba obserwacji z $X=x$, a $n(x,1)$ -- liczba tych spośród nich, w których $Y=1$
(por. Ćwiczenie 1.12 c). Stąd $\widehat\eta(x)$ to frakcja jedynek wśród obserwacji z $X=x$.

Oszacowania są obliczane z losowej próby, więc same są losowe: inna próba da inne wartości.
Sprawdzamy, jak daleko są od wartości z tablicy i jak ta odległość zmienia się z rozmiarem próby $n$.
Zwróć uwagę, że $\widehat\eta(x)$ liczymy tylko z około $n\,p(x)$ obserwacji, a nie z całej próby.
Które oszacowania są więc mniej dokładne przy $n=20$: $\widehat p$ czy $\widehat\eta$?

In [ ]:
def oszacuj(X, Y):
    p_hat = np.array([np.mean(X == x) for x in (1, 2, 3)])
    eta_hat = np.array([... for x in (1, 2, 3)])
    return p_hat, eta_hat

for n in [20, 200, 20_000]:
    X, Y = losuj(n)
    p_hat, eta_hat = oszacuj(X, Y)
    print(f"n = {n:6d}   p_hat = {np.round(p_hat, 3)}   eta_hat = {np.round(eta_hat, 3)}")

### 1.3. Błędy klasyfikatorów na próbie

Klasyfikator dla $x=1,2,3$ zapisujemy jako tablicę klas, np. `np.array([0, 1, 1])`; jego predykcje
na próbie to `klasyfikator[X - 1]`. Błąd na próbie to frakcja obserwacji, w których predykcja różni się od $Y$.
Porównaj wyniki z punktami b), c) i d) z tablicy.

In [ ]:
regula = (eta > 0.5).astype(int)    # reguła z punktu b)
stala = np.array([1, 1, 1])         # klasyfikator zawsze wskazujący klasę 1

X, Y = losuj(10_000)
print("błąd reguły:      ", ...)
print("błąd stałego:     ", ...)
print("P(Y=1) z próby:   ", ...)

### 1.4. Reguła wyuczona z próby

Teraz udajemy, że nie znamy $\eta$. Z małej próby szacujemy $\widehat\eta$ i budujemy regułę
$\widehat\eta(x)>1/2$. Porównujemy dwa jej błędy: na tej samej próbie, z której ją wyznaczyliśmy,
i prawdziwe prawdopodobieństwo błędu, które liczymy z rozkładu, tak jak na tablicy.

Przy małym $n$ może się zdarzyć, że jakaś wartość $x$ nie wystąpi w próbie. Wtedy $\widehat\eta(x)$ wychodzi
`nan` (z ostrzeżeniem), a reguła wskazuje dla tego $x$ klasę 0.

In [ ]:
def blad_prawdziwy(klasyfikator):
    # przy x błąd to eta(x), gdy wskazujemy klasę 0, i 1 - eta(x), gdy klasę 1; ważymy przez p(x)
    return np.sum(p_x * np.where(klasyfikator == 1, 1 - eta, eta))

print(blad_prawdziwy(regula), blad_prawdziwy(stala))   # te same liczby co na tablicy

In [ ]:
n = 20
for powt in range(10):
    X, Y = losuj(n)
    _, eta_hat = oszacuj(X, Y)
    regula_hat = ...      # reguła wyuczona: klasa 1 tam, gdzie eta_hat > 1/2
    blad_proba = ...      # błąd regula_hat na tej samej próbie
    print(regula_hat, round(blad_proba, 3), round(blad_prawdziwy(regula_hat), 3))

Powtórz doświadczenie 1000 razy i oblicz: jak często reguła wyuczona różni się od reguły z tablicy,
średni błąd na próbie i średni błąd prawdziwy. Który z dwóch błędów jest systematycznie mniejszy i dlaczego?
Jak zmieniają się wyniki dla $n=50$ i $n=500$?

In [ ]:
...

## Część 2. Urny (Ćwiczenie 1.12 b)

Kodowanie: urna `0` to U1, `1` to U2; kolor `0` to biała, `1` to czarna.
Tym razem rozkład jest zadany jako $p(y)\,p(x\mid y)$, więc losujemy w odwrotnej kolejności niż w części 1:
najpierw urnę, potem kolor kuli z prawdopodobieństwem zależnym od urny.

In [ ]:
def symuluj(N):
    urna = ...    # 1 (U2) z prawdopodobieństwem 0,6
    kolor = ...   # 1 (czarna) z prawdopodobieństwem 2/7 w U1 i 1/4 w U2
    return urna, kolor

Dla $N=100$, $10^4$ i $10^6$ oszacuj z próby tabelę $p(x,y)$ (wiersze: urny, kolumny: kolory)
oraz $P(Y=\text{U2}\mid X=\text{czarna})$ i porównaj z wynikami z tablicy.

In [ ]:
...

## Część 3. Ryzyko empiryczne a ryzyko (Ćwiczenie 1.14)

Model: $Y=\sin(2\pi X)+\varepsilon$, $X$ jednostajny na $[0,1]$, $\varepsilon\sim N(0,\,0{,}3^2)$.
Ryzyko przybliżamy średnią stratą kwadratową na dużej nowej próbie.

In [ ]:
from numpy.polynomial import Polynomial

sigma = 0.3

def generuj(n):
    X = rng.uniform(0, 1, size=n)
    Y = ...
    return X, Y

X_tr, Y_tr = generuj(30)
X_te, Y_te = generuj(100_000)

In [ ]:
stopnie = range(1, 13)
ryzyko_emp, ryzyko = [], []
for d in stopnie:
    f = Polynomial.fit(X_tr, Y_tr, deg=d)   # f(x) liczy wartość wielomianu
    ryzyko_emp.append(...)                  # średnia strata na zbiorze treningowym
    ryzyko.append(...)                      # średnia strata na nowej próbie

In [ ]:
plt.plot(stopnie, ryzyko_emp, "o-", label="ryzyko empiryczne (trening)")
plt.plot(stopnie, ryzyko, "o-", label="ryzyko (nowa próba)")
plt.axhline(..., color="gray", linestyle="--", label="błąd nieredukowalny")
plt.yscale("log")
plt.xlabel("stopień wielomianu")
plt.ylabel("średnia strata kwadratowa")
plt.legend()
plt.show()

Odpowiedz na pytania a), b) i c) z ćwiczenia. Dla punktu c) zmień rozmiar próby treningowej na 300 i uruchom ponownie trzy ostatnie komórki.